# Prediksi Under-Reimbursement Klaim INA-CBGs pada Pasien ICU Dewasa

**Notebook pengendali seluruh pipeline analisis.**
Penelitian: memprediksi apakah klaim INA-CBGs satu episode rawat akan lebih kecil daripada
biaya riil rumah sakit (*under-reimbursement*), hanya dengan variabel klinis dan administratif
yang tercatat pada **24 jam pertama** perawatan ICU.

---

### Cara memakai notebook ini

| Yang ingin Anda lakukan | Caranya |
|---|---|
| Menjalankan semua dari nol | Menu **Run → Run All Cells**. Sekali klik, seluruh tahap berjalan berurutan. |
| Menjalankan satu tahap saja | Jalankan sel *Setup* di bawah, lalu sel tahap yang diinginkan. |
| Mengubah logika analisis | Edit file `pipelines/<tahap>/*.py`, lalu jalankan ulang sel tahap itu. Berkat `autoreload`, kernel tidak perlu di-restart. |
| Mengubah angka/pengaturan | Edit `config/config.yaml` (jumlah sampel, grid hyperparameter, tanggal split, dll). |
| Menjalankan tanpa notebook | `python pipelines/04_train_model/main.py` — hasilnya identik. |

### Alur tahapan

```
00 data_generation  →  01 eda  →  02 preprocessing  →  03 feature_building  →  04 train_model
   data dummy          lihat        bersihkan &          bentuk matriks         latih, uji,
                       apa adanya   tetapkan outcome     fitur model            jelaskan (SHAP)
```

Setiap tahap membaca file yang ditulis tahap sebelumnya, jadi urutannya penting saat pertama kali dijalankan.

## Setup — menyiapkan lingkungan kerja

Sel ini menyambungkan notebook ke folder proyek dan memuat konfigurasi.
`autoreload` membuat setiap perubahan pada file pipeline langsung terpakai tanpa restart kernel.

> Jalankan sel ini lebih dulu, apa pun tahap yang ingin Anda kerjakan.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Temukan akar folder data_analysis, baik notebook dijalankan dari sini maupun dari subfolder
ROOT = Path.cwd()
while not (ROOT / "pipelines").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from common import load_config, run_pipeline, show_result, show_figures, show_tables
from common.display import show_file

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

cfg = load_config()
print("Folder proyek :", ROOT)
print("Mode data     :", cfg["project"]["data_mode"], "(DUMMY = data sintetis, bukan data pasien)")
print("Seed acak     :", cfg.seed)
print("Jumlah episode dummy :", cfg["data_generation"]["n_episodes"])

### Mengintip pengaturan yang paling sering diubah

Kalau ingin mengubah salah satunya, edit `config/config.yaml` lalu jalankan ulang sel *Setup*.
Untuk percobaan cepat, Anda juga bisa menimpanya langsung di sel berikut (perubahan hanya berlaku di sesi ini).

In [ ]:
# Contoh menimpa pengaturan hanya untuk sesi notebook ini (hapus tanda # untuk memakai):
# cfg.to_dict()["data_generation"]["n_episodes"] = 2000   # perbesar data dummy
# cfg.to_dict()["training"]["outer_folds"] = 3            # cross-validation lebih cepat

pengaturan = {
    "Jumlah episode dummy": cfg["data_generation"]["n_episodes"],
    "Periode data": f"{cfg['data_generation']['study_start']} s.d. {cfg['data_generation']['study_end']}",
    "Batas minimal lama rawat ICU": f"{cfg['preprocessing']['min_icu_hours']} jam",
    "Tanggal pemisah latih/uji": cfg["training"]["temporal_split_date"],
    "Nested CV": f"{cfg['training']['outer_folds']} outer x {cfg['training']['inner_folds']} inner",
    "Target AUC": cfg["training"]["auc_target"],
}
pd.DataFrame({"Pengaturan": list(pengaturan), "Nilai": [str(v) for v in pengaturan.values()]})

---
## Tahap 00 — Membuat data dummy

**Kenapa ada tahap ini?** Data RSCM yang sebenarnya belum tersedia, jadi kita membuat data tiruan yang
strukturnya sama persis dengan Tabel 3.1 proposal: 14 variabel prediktor 24 jam pertama, total tagihan
rumah sakit, dan nilai klaim INA-CBGs per episode.

Data dibuat mengikuti logika sebab-akibat yang nyata — pasien lebih berat → lebih banyak alat penunjang →
lama rawat lebih panjang → biaya membengkak, sementara klaim INA-CBGs bersifat paket dan tidak ikut naik.
Selisih itulah yang nanti diprediksi model.

Sengaja dititipkan juga "data kotor" (duplikat, lama rawat < 24 jam, nilai mustahil, data hilang) supaya
tahap pembersihan benar-benar bekerja.

**Hasil:** `data/raw/icu_inacbg_raw.csv` + kamus data.

> Kalau data RSCM sudah ada: letakkan CSV-nya di `data/raw/` dan **lewati tahap ini**, mulai dari tahap 01.

In [ ]:
hasil_00 = run_pipeline("00_data_generation", cfg=cfg)
show_result(hasil_00)

---
## Tahap 01 — EDA: melihat data apa adanya

Sebelum data disentuh sama sekali, kita lihat dulu bentuk aslinya. Tahap ini menjawab pertanyaan dasar:
seberapa banyak data yang hilang dan pada variabel mana, bagaimana sebaran tiap variabel, berapa persen
episode yang mengalami under-reimbursement, subkelompok mana yang paling terdampak, dan apakah polanya
stabil dari bulan ke bulan.

Semua keputusan pembersihan pada tahap berikutnya harus lahir dari apa yang terlihat di sini — bukan sebaliknya.

**Hasil:** 8 gambar eksplorasi + Tabel 4.1 (karakteristik subjek menurut status outcome).

In [ ]:
hasil_01 = run_pipeline("01_eda", cfg=cfg)
show_result(hasil_01)

Ingin melihat satu tabel saja dengan lebih banyak baris? Contoh untuk Tabel 4.1:

In [ ]:
show_tables(hasil_01, max_rows=25, only=["Tabel 4.1 Karakteristik subjek"])

---
## Tahap 02 — Preprocessing: membersihkan dan menetapkan outcome

Di sini data mentah diubah menjadi **kohort analisis**:

1. menghitung variabel turunan dari timestamp (durasi pra-ICU, lama rawat ICU) dan rasio SpO₂/FiO₂;
2. memeriksa rentang nilai wajar — nilai mustahil (misal MAP 5 mmHg) dijadikan *hilang*, bukan dibuang barisnya;
3. menerapkan kriteria inklusi & eksklusi sesuai proposal → menghasilkan **Gambar 3.1 Alur Seleksi Subjek**;
4. menetapkan outcome di akhir episode: biner (rasio klaim/tagihan < 1) dan magnitudo (log rasio);
5. memberi label periode latih/uji untuk validasi temporal.

**Yang sengaja TIDAK dilakukan di sini:** imputasi, encoding, dan standardisasi. Ketiganya harus terjadi
di dalam fold validasi (tahap 04) agar tidak ada informasi data uji yang bocor ke data latih.

**Hasil:** `data/interim/analytic_cohort.csv`.

In [ ]:
hasil_02 = run_pipeline("02_preprocessing", cfg=cfg)
show_result(hasil_02)

---
## Tahap 03 — Feature building: menyiapkan bahan untuk model

Kohort yang sudah bersih diubah menjadi matriks fitur. Yang terjadi di sini murni transformasi
deterministik — tidak ada yang "dipelajari" dari data:

- **skor komponen mSOFA** (respirasi, kardiovaskular, SSP, ginjal) dari MAP, SpO₂/FiO₂, GCS, dan kreatinin;
- **jumlah organ support** 24 jam pertama (0–3);
- **log durasi pra-ICU**, karena sebarannya sangat miring ke kanan;
- **penanda GCS tidak dapat dinilai** karena pasien tersedasi — informasinya dipertahankan, tidak dibuang.

Tahap ini juga menetapkan peran tiap kolom (numerik / ordinal / biner / kategorik) dan memeriksa
multikolinearitas lewat VIF. Objek preprocessor (MICE + one-hot + standardisasi) hanya *dirakit* di sini,
belum di-fit.

**Hasil:** `data/processed/X_features.csv`, `y_target.csv`, dan `feature_spec.json`.

In [ ]:
hasil_03 = run_pipeline("03_feature_building", cfg=cfg)
show_result(hasil_03)

---
## Tahap 04 — Melatih, menguji, dan menjelaskan model

Tahap terberat sekaligus inti penelitian. Tiga model dibandingkan: **XGBoost**, **Random Forest**, dan
**Elastic Net Logistic Regression** sebagai pembanding yang dioptimalkan setara.

Dua skema validasi dijalankan:

- **Nested cross-validation 5×3** — hyperparameter dipilih di inner fold, performa dinilai di outer fold,
  sehingga estimasinya tidak optimistis;
- **Validasi temporal** — dilatih pada admisi Mei–Des 2025, diuji pada Jan–Mei 2026, untuk menguji
  kestabilan model terhadap pergeseran waktu.

Yang dinilai bukan hanya diskriminasi (AUC-ROC, PR-AUC, sensitivitas, spesifisitas, F1) tetapi juga
**kalibrasi** (Brier score, calibration intercept & slope) — sebab model yang dipakai untuk perencanaan
biaya harus jujur soal besaran probabilitasnya, bukan sekadar bisa mengurutkan pasien.

Setelah itu model terbaik dibedah dengan **SHAP**: variabel mana yang paling berkontribusi, ke arah mana,
bagaimana bentuk hubungannya, bagaimana penjelasan pada pasien per orang, dan interaksi antarvariabel.
Sebagai pembanding metode, dihitung juga **permutation importance**.

⏱️ Perkiraan waktu: 1–2 menit untuk 1.000 episode. Ingin cepat tanpa SHAP? Pakai `skip_shap=True`.

**Hasil:** Tabel 4.2, Tabel 4.3, 13 gambar, dan model tersimpan di `outputs/models/`.

In [ ]:
hasil_04 = run_pipeline("04_train_model", cfg=cfg)          # tambahkan skip_shap=True untuk versi cepat
show_result(hasil_04, tables=False, figures=False)          # ringkasan dulu, tabel & gambar menyusul di bawah

### Tabel 4.2 — Perbandingan performa antarmodel

Baris teratas adalah model dengan AUC nested cross-validation tertinggi.

In [ ]:
show_tables(hasil_04, max_rows=10, only=["Tabel 4.2 Perbandingan performa model"])

### Gambar evaluasi: ROC, precision-recall, kalibrasi, kestabilan antar-fold, matriks konfusi, dan decision curve

In [ ]:
show_figures(hasil_04, only=[
    "01_kurva_roc", "02_kurva_pr", "03_kalibrasi",
    "04_auc_per_fold", "05_matriks_konfusi", "06_decision_curve",
])

### Tabel 4.3 — Peringkat kontribusi prediktor (SHAP)

`mean_abs_shap` = rerata besar kontribusi variabel terhadap prediksi (satuan log-odds).
Kolom `arah_pengaruh` membantu membaca: apakah nilai tinggi menaikkan atau menurunkan risiko under-reimbursement.

In [ ]:
show_tables(hasil_04, max_rows=20, only=["Tabel 4.3 Peringkat kontribusi SHAP"])

### Gambar interpretabilitas

- **Beeswarm** — sebaran kontribusi tiap variabel pada seluruh pasien uji;
- **Importance bar** — peringkat global rerata |SHAP|;
- **Dependence** — bentuk hubungan (linear? ada ambang?) variabel teratas;
- **Waterfall** — penjelasan untuk satu pasien: kenapa pasien *ini* diprediksi berisiko;
- **Interaksi** — pasangan variabel yang pengaruhnya saling bergantung (analisis eksploratif, tujuan khusus 3);
- **Permutation importance** — pembanding metode; peringkat yang sejalan memperkuat temuan.

In [ ]:
show_figures(hasil_04, only=[
    "07_shap_beeswarm", "08_shap_importance", "09_shap_dependence",
    "10_shap_waterfall_risiko_tertinggi", "10_shap_waterfall_risiko_terendah",
    "11_shap_interaksi", "12_permutation_importance",
])

---
## Ringkasan akhir — apa yang sudah dihasilkan

Sel di bawah mengumpulkan angka-angka kunci dari seluruh tahap dan menunjukkan lokasi file hasil.
Bagian ini yang biasanya disalin ke Bab 4 laporan.

In [ ]:
ringkasan = []
for hasil in [hasil_00, hasil_01, hasil_02, hasil_03, hasil_04]:
    for keterangan, nilai in hasil["summary"].items():
        ringkasan.append({"tahap": hasil["stage"], "keterangan": keterangan, "nilai": str(nilai)})

pd.DataFrame(ringkasan)

In [ ]:
print("Semua hasil tersimpan di:")
print("  Gambar :", cfg.path("figures"))
print("  Tabel  :", cfg.path("tables"))
print("  Model  :", cfg.path("models"))
print("  Laporan:", cfg.path("reports"))
print()
n_gambar = sum(1 for _ in cfg.path("figures").rglob("*.png"))
n_tabel = sum(1 for _ in cfg.path("tables").rglob("*.csv"))
print(f"Total {n_gambar} gambar dan {n_tabel} tabel siap dipakai untuk laporan.")

---
### Catatan penting

- Seluruh angka di notebook ini berasal dari **data sintetis**. Tidak ada data pasien nyata.
  Angka performa model di sini **bukan** hasil penelitian dan tidak boleh dilaporkan sebagai temuan.
- Saat data RSCM tersedia: letakkan CSV di `data/raw/`, ubah `project.data_mode` menjadi `REAL` di
  `config/config.yaml`, sesuaikan nama kolom bila berbeda, lalu jalankan tahap 01 ke atas.
- Data pasien nyata tidak boleh ikut ter-commit ke git — lihat pola nama file yang sudah diblokir
  di `.gitignore`.